# 🔍 Debug + Fix — Tìm đủ 222 bài kinh

Chạy **ô 1→4** để tìm nguyên nhân mất bài.  
Sau đó chạy **ô 5→8** để export CSV đầy đủ.

## 1. Setup

In [ ]:
# Cài đặt các thư viện cần thiết (chạy 1 lần nếu chưa cài)
# !pip install beautifulsoup4 lxml
import zipfile, os, re, copy
from bs4 import BeautifulSoup
import pandas as pd

# Đường dẫn file zip chứa các file HTML đã giải nén sẵn trên máy local
ZIP_NAME = 'Hantu.zip'
EXTRACT_DIR = './html_input/Hantu'

# Giải nén file zip nếu thư mục chưa tồn tại
if not os.path.exists(EXTRACT_DIR):
    with zipfile.ZipFile(ZIP_NAME, 'r') as z:
        z.extractall(EXTRACT_DIR)
    print(f'Đã giải nén {ZIP_NAME} vào {EXTRACT_DIR}')
else:
    print(f'Thư mục {EXTRACT_DIR} đã tồn tại, bỏ qua giải nén.')

# Thu thập danh sách file HTML trong thư mục
html_files = []
for root, dirs, fs in os.walk(EXTRACT_DIR):
    for f in sorted(fs):
        if f.lower().endswith('.html'):
            html_files.append(os.path.join(root, f))
html_files.sort()
print(f'✅ {len(html_files)} files')

: 

## 2. DEBUG — Tìm bài kinh bị miss

Kiểm tra toàn bộ các `<p class='head'>` trong tất cả files để xem pattern nào không được parse.

In [ ]:
# Thu thập TẤT CẢ thẻ head trong toàn bộ files
all_heads = []

for path in html_files:
    fname = os.path.basename(path)
    with open(path, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f.read(), 'lxml')

    for p in soup.find_all('p', class_='head'):
        level = p.get('data-head-level', '?')
        # Lấy text thô (không xóa gì)
        raw = p.get_text(strip=True)
        # Xóa thẻ con để lấy text sạch
        h = copy.copy(p)
        for el in h.find_all(['a','span']): el.decompose()
        clean = h.get_text(strip=True)
        # Thuộc div nào?
        parent = p.parent
        parent_cls = ' '.join(parent.get('class', [])) if parent else ''

        all_heads.append({
            'file':       fname,
            'level':      level,
            'parent_cls': parent_cls,
            'clean_text': clean[:80],
        })

df_heads = pd.DataFrame(all_heads)
print(f'Tổng thẻ head tìm thấy: {len(df_heads)}')
print(f'\nPhân bổ theo data-head-level:')
print(df_heads['level'].value_counts().to_string())
print(f'\nPhân bổ theo parent class:')
print(df_heads['parent_cls'].value_counts().to_string())

Tổng thẻ head tìm thấy: 0

Phân bổ theo data-head-level:


KeyError: 'level'

In [ ]:
# Chỉ xem các head level=3 (tiêu đề bài kinh)
df_l3 = df_heads[df_heads['level'] == '3'].copy()
print(f'Tổng head level=3 (tiêu đề kinh): {len(df_l3)}')
print(f'\nPhân bổ theo parent class:')
print(df_l3['parent_cls'].value_counts().to_string())

# Các bài không nằm trong div-jing?
not_in_jing = df_l3[~df_l3['parent_cls'].str.contains('div-jing', na=False)]
print(f'\n⚠️  Head level=3 KHÔNG nằm trong div-jing: {len(not_in_jing)}')
if len(not_in_jing) > 0:
    pd.set_option('display.max_colwidth', 80)
    print(not_in_jing.to_string(index=False))

In [ ]:
# Xem pattern số thứ tự nào KHÔNG match regex （X）
HAN = {'一':1,'二':2,'三':3,'四':4,'五':5,
       '六':6,'七':7,'八':8,'九':9,'十':10,'百':100,'千':1000}

def han_to_int(s):
    if not s: return None
    if re.fullmatch(r'\d+', s): return int(s)
    result = tmp = 0
    for ch in s:
        v = HAN.get(ch)
        if v is None: return None
        if v == 1000: result += (tmp or 1)*1000; tmp = 0
        elif v == 100: result += (tmp or 1)*100; tmp = 0
        elif v == 10:  tmp = (tmp or 1)*10
        else:          tmp += v
    return result + tmp

no_match = []
for _, row in df_l3.iterrows():
    txt = row['clean_text']
    m = re.match(r'^[（(]([0-9一二三四五六七八九十百千]+)[）)]', txt)
    if not m:
        no_match.append(row)

print(f'⚠️  Head level=3 không match pattern （X）: {len(no_match)}')
for r in no_match:
    print(f'  [{r["file"]}] parent={r["parent_cls"]:20s} | "{r["clean_text"][:70]}"')

In [ ]:
# Xem số thứ tự các bài đã parse được — tìm khoảng trống
parsed_nos = []
for _, row in df_l3.iterrows():
    txt = row['clean_text']
    m = re.match(r'^[（(]([0-9一二三四五六七八九十百千]+)[）)]', txt)
    if m:
        n = han_to_int(m.group(1))
        if n: parsed_nos.append(n)

parsed_nos_set = set(parsed_nos)
print(f'Số bài parse được: {len(parsed_nos_set)}')
print(f'Min: {min(parsed_nos_set)}, Max: {max(parsed_nos_set)}')

missing = sorted(set(range(1, max(parsed_nos_set)+1)) - parsed_nos_set)
print(f'\n❌ Số thứ tự BỊ THIẾU ({len(missing)} bài):')
print(missing)

# Bài nào xuất hiện nhiều hơn 1 lần (trùng số)?
from collections import Counter
dup = {k:v for k,v in Counter(parsed_nos).items() if v > 1}
if dup:
    print(f'\n⚠️  Số thứ tự xuất hiện > 1 lần: {dup}')

## 3. DEBUG — Xem raw HTML của bài bị thiếu

In [ ]:
# Tìm đúng file chứa bài bị miss và in HTML xung quanh
# (chạy sau khi biết missing list từ ô trên)

# Ví dụ nếu missing = [206, 207, ...]
# Tìm trong df_l3 xem bài 205 và 208 ở file nào
TARGET_NOS = missing  # tự động từ ô trên

if TARGET_NOS:
    # Tìm file chứa các bài ngay trước số bị miss
    before_no = min(TARGET_NOS) - 1
    rows_before = []
    for _, row in df_l3.iterrows():
        txt = row['clean_text']
        m = re.match(r'^[（(]([0-9一二三四五六七八九十百千]+)[）)]', txt)
        if m and han_to_int(m.group(1)) == before_no:
            rows_before.append(row)

    print(f'Bài trước số bị miss (bài {before_no}) ở file:')
    for r in rows_before:
        print(f'  {r["file"]}  parent={r["parent_cls"]}')
        # In 200 ký tự HTML xung quanh
        fpath = [p for p in html_files if os.path.basename(p) == r['file']]
        if fpath:
            with open(fpath[0], 'r', encoding='utf-8') as f:
                raw_html = f.read()
            # Tìm vị trí text
            idx = raw_html.find(txt[:20])
            if idx > 0:
                snippet = raw_html[idx:idx+600]
                print(f'\n--- HTML snippet ---')
                print(snippet)
                print('---')
else:
    print('✅ Không có bài bị thiếu!')

## 4. Parser đã fix — hỗ trợ mọi cấu trúc

In [ ]:
def get_text(p_tag):
    """Lấy text sạch từ <p> — xóa lineInfo, doube-line-note, noteAnchor"""
    tag = copy.copy(p_tag)
    for el in tag.find_all(True):
        cls  = el.get('class', [])
        href = el.get('href', '')
        if ('lineInfo'        in cls or
            'doube-line-note' in cls or
            'noteAnchor'      in cls or
            href.startswith('#n')):
            el.decompose()
    txt = tag.get_text(separator='')
    txt = re.sub(r'[ \t]+', ' ', txt).strip()
    txt = re.sub(r'\n+', ' ', txt).strip()
    return txt


def parse_head(head_tag):
    """Parse tiêu đề kinh → (sutta_no, sutta_name)"""
    if not head_tag: return None, ''
    h = copy.copy(head_tag)
    for el in h.find_all(['a','span']): el.decompose()
    raw = h.get_text(strip=True)
    # Pattern 1: （一百二十五）, （一）, (1)
    m = re.match(r'^[（(]([0-9一二三四五六七八九十百千]+)[）)](.+)$', raw)
    if m:
        no   = han_to_int(m.group(1))
        name = re.sub(r'^.+品', '', m.group(2).strip())
        return no, (name or m.group(2).strip())
    return None, raw


def extract_jing_content(container, sutta_no, sutta_name):
    """
    Từ bất kỳ container nào (div-jing hoặc khác),
    lấy tất cả <p> không phải head và trả về rows.
    """
    rows      = []
    group     = 0
    idx       = 0
    last_page = None

    for p in container.find_all('p', recursive=True):
        if 'head' in p.get('class', []):
            continue

        ls = p.find('span', class_='lineInfo')
        if ls and ls.get('line'):
            page = ls['line'][:4]
            if page != last_page:
                group    += 1
                idx       = 0
                last_page = page
        if group == 0:
            group = 1

        content = get_text(p)
        if not content:
            continue
        if re.search(r'竟', content) and len(content) < 25:
            continue

        idx += 1
        rows.append({
            'Sutta_No':   sutta_no,
            'Sutta_Name': sutta_name,
            'Segment_ID': f'MA.{sutta_no}.{group}.{idx}',
            'Content':    content,
        })
    return rows


def parse_file_v2(html_path):
    """
    Parser v2 — tìm kinh theo MỌI cấu trúc:
    1. div.div-jing (chuẩn)
    2. Bất kỳ container nào có <p class='head' data-head-level='3'>
    """
    with open(html_path, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f.read(), 'lxml')

    rows      = []
    seen_nos  = set()  # tránh đếm trùng

    # ── Chiến lược 1: div.div-jing (chuẩn) ──
    for div in soup.find_all('div', class_='div-jing'):
        head = div.find('p', class_='head')
        no, name = parse_head(head)
        if no is None: no = 0
        seen_nos.add((no, name))
        rows.extend(extract_jing_content(div, no, name))

    # ── Chiến lược 2: head level=3 không trong div-jing ──
    # Tìm tất cả <p class='head' data-head-level='3'>
    # mà parent KHÔNG phải div-jing
    for head in soup.find_all('p', class_='head', attrs={'data-head-level': '3'}):
        parent = head.parent
        parent_cls = parent.get('class', []) if parent else []
        if 'div-jing' in parent_cls:
            continue  # đã xử lý ở chiến lược 1

        no, name = parse_head(head)
        if no is None: no = 0
        if (no, name) in seen_nos:
            continue  # tránh trùng
        seen_nos.add((no, name))

        # Lấy container = parent của head tag
        rows.extend(extract_jing_content(parent, no, name))

    return rows


print('✅ Parser v2 sẵn sàng')

## 5. Chạy toàn bộ với parser v2

In [ ]:
all_rows = []
errors   = []

for i, path in enumerate(html_files, 1):
    fname = os.path.basename(path)
    try:
        rows = parse_file_v2(path)
        all_rows.extend(rows)
        print(f'[{i:02d}/{len(html_files)}] {fname:25s} → {len(rows):4d} segments')
    except Exception as e:
        errors.append(fname)
        print(f'[{i:02d}] ⚠️  LỖI {fname}: {e}')

print(f'\n✅ Xong — {len(all_rows):,} segments')
if errors:
    print('⚠️  Files lỗi:', errors)

## 6. Gán Sutta_No toàn corpus + kiểm tra

In [ ]:
df = pd.DataFrame(all_rows)

# Gán Global_No liên tục theo thứ tự xuất hiện
keys = (
    df[['Sutta_No','Sutta_Name']]
    .drop_duplicates()
    .reset_index(drop=True)
)
keys['Global_No'] = keys.index + 1

df = df.merge(keys, on=['Sutta_No','Sutta_Name'], how='left')
df['Sutta_No'] = df['Global_No']
df = df.drop(columns=['Global_No'])

# Cập nhật Segment_ID
def fix_seg(row):
    parts = row['Segment_ID'].split('.')
    parts[1] = str(row['Sutta_No'])
    return '.'.join(parts)
df['Segment_ID'] = df.apply(fix_seg, axis=1)
df = df[['Sutta_No','Sutta_Name','Segment_ID','Content']]

total_sutras = df['Sutta_No'].nunique()
print(f'Tổng segments : {len(df):,}')
print(f'Tổng kinh     : {total_sutras}')

# Kiểm tra số nào còn thiếu
all_nos  = set(df['Sutta_No'].unique())
expected = set(range(1, 223))  # 1→222
missing  = sorted(expected - all_nos)
extra    = sorted(all_nos - expected)

if missing:
    print(f'\n❌ Vẫn còn thiếu {len(missing)} bài: {missing}')
    print('→ Chạy ô DEBUG bên trên để xem raw HTML của bài đó')
else:
    print('\n✅ Đủ 222 bài kinh!')
if extra:
    print(f'⚠️  Số dư ngoài 1-222: {extra}')

In [ ]:
# Xem danh sách tên 222 kinh
sutra_list = df[['Sutta_No','Sutta_Name']].drop_duplicates().sort_values('Sutta_No')
print(f'{'No':>4}  {'Sutta_Name'}')
print('-'*40)
for _, r in sutra_list.iterrows():
    print(f'{r["Sutta_No"]:>4}  {r["Sutta_Name"]}')

## 7. Lưu & Download

In [ ]:
OUTPUT = '/content/MA_segments.csv'
df.to_csv(OUTPUT, index=False, encoding='utf-8-sig')

print(f'✅ Đã lưu : {OUTPUT}')
print(f'   Size   : {os.path.getsize(OUTPUT)//1024} KB')
print(f'   Rows   : {len(df):,}')
print(f'   Sutras : {df["Sutta_No"].nunique()}')

files.download(OUTPUT)
print('⬇️  Đang tải về...')

In [ ]:
import os, re, copy, zipfile, sys
import pandas as pd
from bs4 import BeautifulSoup
from google.colab import files

# Tăng giới hạn đệ quy để xử lý file lồng nhau sâu (như file 003, 011)
sys.setrecursionlimit(5000)

# --- 1. CÁC HÀM HỖ TRỢ (Định nghĩa đầy đủ tại đây) ---
HAN = {'一':1,'二':2,'三':3,'四':4,'五':5,'六':6,'七':7,'八':8,'九':9,'十':10,'百':100,'千':1000,'〇':0,'零':0}

def han_to_int(s):
    if not s: return None
    if re.fullmatch(r'\d+', s): return int(s)
    # Kiểu đếm (一七二 -> 172)
    if all(c in HAN for c in s) and '十' not in s and '百' not in s and len(s) > 1:
        res = 0
        for char in s: res = res * 10 + HAN[char]
        return res
    # Kiểu văn bản (一百七十二 -> 172)
    result = tmp = 0
    for ch in s:
        v = HAN.get(ch)
        if v is None: continue
        if v == 1000: result += (tmp or 1)*1000; tmp = 0
        elif v == 100: result += (tmp or 1)*100; tmp = 0
        elif v == 10:  tmp = (tmp or 1)*10
        else:          tmp = v
    return result + tmp

def get_pure_text(p_tag):
    """Lấy nội dung kinh văn sạch tuyệt đối ngay trong cấu trúc HTML"""
    tag = copy.copy(p_tag)
    # Xóa các thành phần rác của CBETA (mã trang, chú thích ẩn...)
    for el in tag.find_all(True):
        cls = el.get('class', [])
        cls_list = cls if isinstance(cls, list) else [str(cls)]
        href = el.get('href', '')
        if (any(c in ['lineInfo', 'lineinfo', 'noteAnchor', 'anchor', 'doube-line-note', 't', 'bglab'] for c in cls_list)
            or (isinstance(href, str) and href.startswith('#n'))):
            el.decompose()

    txt = tag.get_text(separator='').strip()
    # Xóa mã hiệu trang p0421... và chú thích Latin/Số
    txt = re.sub(r'\[[a-zA-Z0-9]+\]|p[0-9]{4}[a-z][0-9]{2}|QC[a-zA-Z0-9_]+', '', txt)
    # Lọc dòng chú thích dị bản
    if sum(1 for ed in ["【大】", "【宋】", "【元】", "【明】", "【麗】"] if ed in txt) >= 2:
        return ""
    return re.sub(r'\s+', ' ', txt).strip()

def parse_ma_ultimate(html_path):
    with open(html_path, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f.read(), 'html.parser')
    rows = []
    # Tìm mọi thẻ có thể chứa chữ
    all_elements = soup.find_all(['p', 'div', 'span', 'h3'])
    current_no, current_name, para_idx = None, "", 1

    for el in all_elements:
        raw_txt = el.get_text(strip=True)
        if len(raw_txt) < 2: continue

        # Nhận diện bài kinh mới
        clean_search = re.sub(r'\[.*?\]', '', raw_txt)
        m = re.search(r'^[（(\[](?:第)?([0-9一二三四五六七八九十百千〇零]+)[）)\]]', clean_search)

        if m:
            new_no = han_to_int(m.group(1))
            if new_no is not None and 1 <= new_no <= 222:
                current_no, para_idx = new_no, 1
                name_part = raw_txt.split('）')[-1].split(')')[-1].strip()
                current_name = name_part if name_part else f"MA {new_no}"
                continue

        # Lưu nội dung
        if current_no is not None:
            content = get_pure_text(el)
            if content and len(content) > 10:
                # Lọc rác hành chính và Uddana (mục lục)
                if any(x in content for x in ['校注', '經文資訊', 'No.', '卷第', '第一誦', '筆受', '譯']): continue
                if content.count('、') > 4 and len(content) < 100: continue
                if '竟' in content[:10] and len(content) < 30: continue

                rows.append({'Raw_No': current_no, 'Sutta_Name': current_name,
                             'Segment_ID': f'MA.{current_no}.{para_idx}', 'Content': content})
                para_idx += 1

    seen = set()
    return [r for r in rows if not (r['Content'] in seen or seen.add(r['Content']))]

# --- 2. QUY TRÌNH THỰC THI CHÍNH ---
EXTRACT_DIR = '/content/hantu_html'
if not os.path.exists(EXTRACT_DIR):
    print("❌ Thư mục hantu_html chưa có. Bạn hãy chạy Cell giải nén Zip trước.")
else:
    html_files = []
    for root, dirs, fs in os.walk(EXTRACT_DIR):
        for f in fs:
            if f.lower().endswith('.html'): html_files.append(os.path.join(root, f))
    html_files.sort()

    print(f'🚀 Đang nội soi và làm sạch 222 bài kinh từ {len(html_files)} file...')
    all_data = []
    for i, path in enumerate(html_files, 1):
        res = parse_ma_ultimate(path)
        all_data.extend(res)
        if i % 10 == 0: print(f"  Đã xử lý {i}/60 file...")

    if not all_data:
        print("❌ Không bóc tách được dữ liệu. Hãy kiểm tra lại file HTML.")
    else:
        df = pd.DataFrame(all_data)
        # Đánh số Global liên tục 1-222
        keys = df[['Raw_No', 'Sutta_Name']].drop_duplicates().reset_index(drop=True)
        keys['Sutta_No'] = keys.index + 1
        df = df.merge(keys, on=['Raw_No', 'Sutta_Name'], how='left')

        # Chuẩn hóa Segment_ID và lọc cột
        df['Segment_ID'] = df.apply(lambda r: f"MA.{r['Sutta_No']}.{r['Segment_ID'].split('.')[-1]}", axis=1)
        df = df[['Sutta_No', 'Sutta_Name', 'Segment_ID', 'Content']]

        print("\n" + "="*40)
        print(f"✅ THÀNH CÔNG!")
        print(f"📊 Tổng số kinh bóc tách: {df['Sutta_No'].nunique()}/222")
        print(f"📝 Tổng số dòng (segments): {len(df)}")
        print("="*40)

        # Xuất file
        OUTPUT = '/content/Hantu_MA_222_Final.csv'
        df.to_csv(OUTPUT, index=False, encoding='utf-8-sig')
        files.download(OUTPUT)

🚀 Đang nội soi và làm sạch 222 bài kinh từ 60 file...
  Đã xử lý 10/60 file...
  Đã xử lý 20/60 file...
  Đã xử lý 30/60 file...
  Đã xử lý 40/60 file...
  Đã xử lý 50/60 file...
  Đã xử lý 60/60 file...

✅ THÀNH CÔNG!
📊 Tổng số kinh bóc tách: 221/222
📝 Tổng số dòng (segments): 7684


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
import re
import pandas as pd
from bs4 import BeautifulSoup
from google.colab import files

# 1. Tìm file Quyển 60 (T0026_060.html)
input_root = "/content/hantu_html"
file_60 = ""
for root, dirs, fs in os.walk(input_root):
    for f in fs:
        if "060" in f and f.endswith('.html'):
            file_60 = os.path.join(root, f)

if not file_60:
    print("❌ Không tìm thấy file T0026_060.html. Hãy đảm bảo bạn đã giải nén đủ 60 file.")
else:
    print(f"🎯 Đang trích xuất bài cuối cùng (222) từ: {file_60}")
    with open(file_60, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f.read(), 'html.parser')

    # Lấy toàn bộ thẻ có khả năng chứa nội dung
    elements = soup.find_all(['p', 'div', 'span'])

    found_222 = False
    recovery_222 = []
    para_idx = 1

    for el in elements:
        txt_raw = el.get_text().strip()
        # Tìm mốc bài 222 (nhận diện cả dấu ngoặc dính rác)
        clean_txt = re.sub(r'\[.*?\]', '', txt_raw)

        if "（二二二）" in clean_txt or "例經" in clean_txt:
            if not found_222:
                found_222 = True
                print("✅ Đã tìm thấy điểm bắt đầu bài 222.")
                continue

        if found_222:
            # Dấu hiệu kết thúc toàn bộ bộ kinh
            if "中阿含經卷" in clean_txt and "終" in clean_txt:
                break

            # Làm sạch rác hệ thống
            content_clean = re.sub(r'\[.*?\]|p[0-9]{4}[a-z][0-9]{2}|QC[a-zA-Z0-9_]+', '', txt_raw).strip()
            text_only = re.sub(r'[^\u4e00-\u9fff]', '', content_clean)

            if len(text_only) > 5:
                # Lọc bỏ dòng tiêu đề quyển 60
                if "卷第六十" in content_clean: continue

                recovery_222.append({
                    'Sutta_No': 222,
                    'Sutta_Name': "例經",
                    'Segment_ID': f'MA.222.{para_idx}',
                    'Content': content_clean
                })
                para_idx += 1

    # 2. Gộp vào dữ liệu hiện có
    if recovery_222:
        df_222 = pd.DataFrame(recovery_222)
        # Loại bỏ trùng lặp nếu bài 222 đã có 1 dòng cũ
        df_clean = df[df['Sutta_No'] != 222]
        df_final = pd.concat([df_clean, df_222], ignore_index=True)
        df_final = df_final.sort_values(by=['Sutta_No', 'Segment_ID'])

        print("\n" + "★"*40)
        print(f"🎊 THÀNH CÔNG TRỌN VẸN: {df_final['Sutta_No'].nunique()}/222 bài.")
        print(f"📝 Tổng số dòng (segments): {len(df_final)}")
        print("★"*40)

        # Lưu file cuối cùng
        FINAL_FILE = 'Hantu_MA_222_Perfect_Full_Final.csv'
        df_final.to_csv(FINAL_FILE, index=False, encoding='utf-8-sig')
        files.download(FINAL_FILE)
    else:
        print("❌ Vẫn không bóc tách được bài 222. Hãy kiểm tra nội dung file T0026_060.html")

🎯 Đang trích xuất bài cuối cùng (222) từ: /content/hantu_html/T0026_060.html
✅ Đã tìm thấy điểm bắt đầu bài 222.

★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
🎊 THÀNH CÔNG TRỌN VẸN: 222/222 bài.
📝 Tổng số dòng (segments): 7716
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import re
import unicodedata
from google.colab import files

# 1. Load file
FILE_PALI = "Pali_Perfect.csv"
df_p = pd.read_csv(FILE_PALI)

# --- BƯỚC VÁ LỖI: TẠO CỘT CLEANED_CONTENT NẾU THIẾU ---
def pali_clean_logic(text):
    if not isinstance(text, str): return ""
    # Chuẩn hóa Unicode và viết thường
    text = unicodedata.normalize('NFC', text).lower()
    # Chỉ giữ lại chữ cái Pali (a-z và các dấu phụ) và dấu cách
    # Regex này giữ lại: ā ī ū ṃ ṅ ñ ṭ ḍ ṇ ḷ
    text = re.sub(r'[^a-zāīūṃṅñṭḍṇḷ\s]', ' ', text)
    # Xóa khoảng trắng thừa
    return re.sub(r'\s+', ' ', text).strip()

if 'Cleaned_Content' not in df_p.columns:
    print("🔄 Đang tạo cột 'Cleaned_Content' cho dữ liệu Pali...")
    # Nếu file của bạn dùng tên cột khác (ví dụ 'text'), hãy đổi 'Content' thành tên đó
    source_col = 'Content' if 'Content' in df_p.columns else df_p.columns[-1]
    df_p['Cleaned_Content'] = df_p[source_col].apply(pali_clean_logic)
    print("✅ Đã tạo xong cột làm sạch.")

# --- TIẾP TỤC BÁO CÁO KIỂM TOÁN ---
print("\n" + "="*50)
print("🔍 BÁO CÁO KIỂM TOÁN DỮ LIỆU PALI (ĐÃ CẬP NHẬT)")
print("="*50)

# 1. Số lượng bài kinh
unique_suttas = sorted(df_p['Sutta_No'].unique())
print(f"1. Tổng số bài kinh: {len(unique_suttas)}/152")

# 2. Kiểm tra rác PTS
pts_pattern = r'\[[M|P|V].*?\]'
pts_rows = df_p[df_p['Content'].str.contains(pts_pattern, na=False)]
print(f"2. Số dòng còn dính mã hiệu PTS: {len(pts_rows)}")

# 3. Kiểm tra rác ngôn ngữ (Ký tự lạ không phải Pali)
# Tìm các từ có chứa x, y, z (Pali chuẩn hầu như không có)
strange_chars = df_p[df_p['Cleaned_Content'].str.contains(r'[xyz]', na=False)]
print(f"3. Số dòng nghi vấn chứa ký tự lạ (x, y, z): {len(strange_chars)}")

# 4. Thống kê độ dài
df_p['word_count'] = df_p['Cleaned_Content'].str.split().str.len()
print(f"4. Độ dài trung bình: {df_p['word_count'].mean():.1f} từ/đoạn")

# 5. Kiểm tra đoạn siêu ngắn (Nghi vấn rác hoặc tiêu đề sót lại)
short_segs = df_p[df_p['word_count'] < 3]
print(f"5. Số đoạn siêu ngắn (<3 từ): {len(short_segs)}")

print("\n" + "="*50)
print("📌 FILE ĐÃ SẴN SÀNG ĐỂ SO SÁNH")
print("="*50)

# Hiển thị 5 dòng ngẫu nhiên để kiểm tra mắt
display(df_p[['Sutta_No', 'Content', 'Cleaned_Content']].sample(5))

# Tùy chọn: Lưu lại file đã có cột Cleaned_Content để dùng luôn cho các bước sau
df_p.to_csv("Pali_Perfect.csv", index=False, encoding='utf-8-sig')
print("\n💾 Đã lưu file sạch 100% vào: Pali_Perfect.csv")
# files.download("Pali_Perfect.csv")


🔍 BÁO CÁO KIỂM TOÁN DỮ LIỆU PALI (ĐÃ CẬP NHẬT)
1. Tổng số bài kinh: 152/152
2. Số dòng còn dính mã hiệu PTS: 0
3. Số dòng nghi vấn chứa ký tự lạ (x, y, z): 3513
4. Độ dài trung bình: 61.2 từ/đoạn
5. Số đoạn siêu ngắn (<3 từ): 93

📌 FILE ĐÃ SẴN SÀNG ĐỂ SO SÁNH


,Sutta_No,Content,Cleaned_Content
3224,118,‘‘So tathāsato viharanto taṃ dhammaṃ paññāya p...,so tathāsato viharanto taṃ dhammaṃ paññāya pav...
1953,65,"‘‘Puna caparaṃ, bhaddāli, bhikkhu sukhassa ca ...",puna caparaṃ bhaddāli bhikkhu sukhassa ca pahā...
784,24,"‘‘No hidaṃ, āvuso’’.",no hidaṃ āvuso
1015,31,"‘‘Kacci vo, anuruddhā, khamanīyaṃ, kacci yāpan...",kacci vo anuruddhā khamanīyaṃ kacci yāpanīyaṃ ...
1528,46,"‘‘Tatra, bhikkhave, yamidaṃ dhammasamādānaṃ pa...",tatra bhikkhave yamidaṃ dhammasamādānaṃ paccup...



💾 Đã lưu file sạch 100% vào: Pali_Perfect.csv
